In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Forward Trespass

In [6]:
# Load the transitions from the forward_tres
import pickle
# Generated by test_tres.py

with open('../data/graph/transitions/LEMD_EGLL_2023_04_01_CLB.pkl', 'rb') as f:
    transitions_list = pickle.load(f)


In [7]:
# transitions_list = [(185, 0, 0.0, 546, 11, 14378.0),
#  (185, 0, 0.0, 472, 8, 11638.0),
#  (185, 0, 0.0, 491, 8, 10883.0),...]
#  (node_id, k_u_std_idx, alt_u_std, v_idx, k_v_std_idx, alt_v_std)

from_ids = set(t[0] for t in transitions_list)
to_ids = set(t[3] for t in transitions_list)
print(f"Number of unique from IDs: {len(from_ids)}")
print(f"Number of unique to IDs: {len(to_ids)}")


Number of unique from IDs: 47
Number of unique to IDs: 145


In [8]:
def print_states_at_node_from_transitions(node_idx: int, transitions_list: list[tuple], idx_to_node: dict[int, str]):
    """
    Print a padded table of all the states with node_id == node_idx from transitions_list.
    transitions_list: list of (u_idx, k_u_std_idx, alt_u_std, v_idx, k_v_std_idx, alt_v_std)
    """
    headers = ["u_wp", "k_u_idx", "alt_u", "v_wp", "k_v_idx", "alt_v"]
    col_widths = [7, 8, 10, 7, 8, 10]
    # Print header
    header_row = "".join(h.ljust(w) for h, w in zip(headers, col_widths))
    print(header_row)
    print("-" * sum(col_widths))
    for t in transitions_list:
        if t[0] == node_idx:
            u_wp = idx_to_node.get(t[0], str(t[0]))
            v_wp = idx_to_node.get(t[3], str(t[3]))
            row = [
                str(u_wp).ljust(col_widths[0]),
                str(t[1]).ljust(col_widths[1]),
                f"{t[2]:.1f}".ljust(col_widths[2]),
                str(v_wp).ljust(col_widths[3]),
                str(t[4]).ljust(col_widths[4]),
                f"{t[5]:.1f}".ljust(col_widths[5]),
            ]
            print("".join(row))

print_states_at_node_from_transitions(node_to_idx['LEMD'], transitions_list, idx_to_node)

NameError: name 'node_to_idx' is not defined

# Backward Trespass

In [9]:
# Load the transitions from the forward_tres
import pickle
# Generated by test_tres.py

with open('../data/graph/transitions/LEMD_EGLL_2023_04_01_CLSR.pkl', 'rb') as f:
    transitions_closure_list = pickle.load(f) # (u_idx, k_u_std_idx, rho_u_std_idx, alt_u_std, phase_u_std, v_idx, k_v_idx, rho_v_idx, alt_v_val, phi_v_idx)


In [10]:
import torch
import numpy as np

import networkx as nx
# Load the route graph
G = nx.read_gml("../data/graph/LEMD_EGLL_2023_04_01.gml")
node_to_idx = {node: i for i, node in enumerate(G.nodes())}
idx_to_node = {i: node for i, node in enumerate(G.nodes())}

def print_states_at_node(node_idx: int, transitions_closure_list: list[tuple[int, int, int, float, int, int, int, int, float, int]],
                         idx_to_node: dict[int, str]):
    # Define column headers and widths, replacing u_idx and v_idx with waypoint names
    headers = [
        "u_wp", "k_u_idx", "rho_u_idx", "alt_u", "phase_u",
        "v_wp", "k_v_idx", "rho_v_idx", "alt_v", "phase_v"
    ]
    col_widths = [7, 8, 10, 10, 8, 7, 8, 10, 10, 8]
    # Phase mapping
    phase_map = {0: "CLB", 1: "CRZ", 2: "DES"}
    # Print header
    header_row = "".join(h.ljust(w) for h, w in zip(headers, col_widths))
    print(header_row)
    print("-" * sum(col_widths))
    # Print each matching transition in padded columns, mapping indices to waypoint names
    for transition in transitions_closure_list:
        if transition[0] == node_idx:
            phase_u_str = phase_map.get(transition[4], str(transition[4]))
            phase_v_str = phase_map.get(transition[9], str(transition[9]))
            u_wp = idx_to_node.get(transition[0], str(transition[0]))
            v_wp = idx_to_node.get(transition[5], str(transition[5]))
            row = (
                f"{u_wp:<7}{transition[1]:<8}{transition[2]:<10}"
                f"{transition[3]:<10.1f}{phase_u_str:<8}"
                f"{v_wp:<7}{transition[6]:<8}{transition[7]:<10}"
                f"{transition[8]:<10.1f}{phase_v_str:<8}"
            )
            print(row)

node_to_inspect = 'LEMD'
print(node_to_inspect)
print_states_at_node(node_to_idx[node_to_inspect], transitions_closure_list, idx_to_node)


LEMD
u_wp   k_u_idx rho_u_idx alt_u     phase_u v_wp   k_v_idx rho_v_idx alt_v     phase_v 
--------------------------------------------------------------------------------------
LEMD   39      0         35000.0   CRZ     NEDUS  43      0         35000.0   CRZ     
LEMD   39      37        0.0       CLB     NEDUS  43      0         35000.0   CRZ     
LEMD   39      37        0.0       CLB     BAKUP  43      0         35000.0   CRZ     
LEMD   39      0         35000.0   CRZ     BAKUP  45      0         35000.0   CRZ     
LEMD   40      0         35000.0   CRZ     BAKUP  46      0         35000.0   CRZ     
LEMD   40      0         35000.0   CRZ     BAKUP  47      0         35000.0   CRZ     
LEMD   39      0         35000.0   CRZ     NUBLO  43      0         35000.0   CRZ     
LEMD   39      37        0.0       CLB     NUBLO  43      0         35000.0   CRZ     
LEMD   40      0         35000.0   CRZ     NUBLO  44      0         35000.0   CRZ     
LEMD   39      0         35000.0   CRZ